In [ ]:
import gzip
import json
import pickle 

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from IPython.display import VimeoVideo
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline


In [ ]:
VimeoVideo("694695674", h="538b4d2725", width=600)


In [ ]:
def wrangle(filename):
    with gzip.open(filename, "r") as f:
        data = json.load(f)
        
    # Load dictionary into DataFrame, set index
    df = pd.DataFrame().from_dict(data["data"]).set_index("company_id")
    
    return df


In [ ]:
df = wrangle("data/poland-bankruptcy-data-2009.json.gz")
print(df.shape)
df.head()


In [ ]:
target = "bankrupt"
X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
VimeoVideo("694695662", h="dc60d76861", width=600)


In [ ]:
over_sampler = RandomOverSampler(random_state=42)
X_train_over, y_train_over = over_sampler.fit_resample(X_train, y_train)
print("X_train_over shape:", X_train_over.shape)
X_train_over.head()


In [ ]:
# Baseline = accuracy of always predicting the most frequent class.
# Baseline Accuracy=max(class proportions) 
#Baseline = majority class accuracy

acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))


In [ ]:
VimeoVideo("694695643", h="32c3d5b1ed", width=600)


In [ ]:
clf = make_pipeline(
    SimpleImputer(),
    RandomForestClassifier(random_state = 42)
)
print(clf)


In [ ]:
VimeoVideo("694695619", h="2c41dca371", width=600)


In [ ]:
cv_acc_scores = cross_val_score(clf, X_train_over,y_train_over, cv=5, n_jobs=-1)
print(cv_acc_scores)


In [ ]:
VimeoVideo("694695593", h="5143f0b63f", width=600)


In [ ]:
params = {
    "simpleimputer__strategy": ["mean","median"],
    "randomforestclassifier__n_estimators" : range(25,100,25),
    "randomforestclassifier__max_depth": range(10,50,10)
}
params


In [ ]:
VimeoVideo("694695574", h="8588bf015f", width=600)


In [ ]:
model = GridSearchCV(
    clf,
    param_grid=params,
    cv=5,
    n_jobs=-1,
    verbose=1
)
model


In [ ]:
VimeoVideo("694695566", h="f4e9910a9e", width=600)


In [ ]:
# Train model
model.fit(X_train_over, y_train_over)


In [ ]:
VimeoVideo("694695546", h="4ae60129c4", width=600)


In [ ]:
cv_results = pd.DataFrame(model.cv_results_)
cv_results.head(10)


In [ ]:
VimeoVideo("694695537", h="e460435664", width=600)


In [ ]:
# Create mask
mask = cv_results["param_randomforestclassifier__max_depth"] == 10
# Plot fit time vs n_estimators
plt.plot(
    cv_results[mask]["param_randomforestclassifier__n_estimators"],
    cv_results[mask]["mean_fit_time"]
)
# Label axes
plt.xlabel("Number of Estimators")
plt.ylabel("Mean Fit Time [seconds]")
plt.title("Training Time vs Estimators (max_depth=10)");


In [ ]:
VimeoVideo("694695525", h="99f2dfc9eb", width=600)


In [ ]:
# Create mask

mask = cv_results["param_randomforestclassifier__n_estimators"] == 25
# Plot fit time vs max_depth
plt.plot(
    cv_results[mask]["param_randomforestclassifier__max_depth"],
    cv_results[mask]["mean_fit_time"]
)

# Label axes
plt.xlabel("Max Depth")
plt.ylabel("Mean Fit Time [seconds]")
plt.title("Training Time vs Max Depth (n_estimators=25)");


In [ ]:
VimeoVideo("694695505", h="f98f660ce1", width=600)


In [ ]:
# Extract best hyperparameters
model.best_params_


In [ ]:
acc_train = model.score(X_train, y_train)
acc_test = model.score(X_test, y_test)

print("Training Accuracy:", round(acc_train, 4))
print("Test Accuracy:", round(acc_test, 4))


In [ ]:
y_test.value_counts()


In [ ]:
VimeoVideo("694695486", h="1d6ac2bf77", width=600)


In [ ]:
# Plot confusion matrix
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)


In [ ]:
VimeoVideo("698358615", h="3fd4b2186a", width=600)


In [ ]:
# Get feature names from training data
features = X_train_over.columns
# Extract importances from model
importances = model.best_estimator_.named_steps["randomforestclassifier"].feature_importances_
# Create a series with feature names and importances
feat_imp = pd.Series(importances, index=features).sort_values()
# Plot 10 most important features
feat_imp.tail(10).plot(kind="barh")

plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("Feature Importance");


In [ ]:
VimeoVideo("694695478", h="a13bdacb55", width=600)


In [ ]:
# Save model
with open("model-5-3.pkl", "wb") as f:
    pickle.dump(model,f)


In [ ]:
VimeoVideo("694695451", h="fc96dd8d1f", width=600)


In [ ]:
def make_predictions(data_filepath, model_filepath):
    # Wrangle JSON file
    X_test = wrangle(data_filepath)
    # Load model
    with open(model_filepath, "rb") as f:
        model = pickle.load(f)
    # Generate predictions
    y_test_pred = model.predict(X_test)
    # Put predictions into Series with name "bankrupt", and same index as X_test
    y_test_pred = pd.Series(y_test_pred, index=X_test.index, name="bankrupt")
    return y_test_pred


In [ ]:
VimeoVideo("694695426", h="f75588d43a", width=600)


In [ ]:
y_test_pred = make_predictions(
    data_filepath="data/poland-bankruptcy-data-2009-mvp-features.json.gz",
    model_filepath="model-5-3.pkl",
)

print("predictions shape:", y_test_pred.shape)
y_test_pred.head(20)
